# YT Agent — Colab GPU worker

Manual companion to the Kaggle GPU worker. Boots a FastAPI backend on a free Colab GPU runtime and connects **outbound** to your Coolify dashboard — no inbound tunnel, no static IP required. Once running it polls `/api/jobs/claim` and heartbeats every 30 sec into the Pocketbase `backends` collection, so the Monitor page detects it within milliseconds.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (free).
2. Open **🔑 Secrets** (left sidebar key icon) and add the **required** secrets:
   - `COOLIFY_BASE_URL` — e.g. `https://yt-agent.thyker.online`
   - `PB_URL` — e.g. `https://yt-agent.thyker.online/pb`
   - `POCKETBASE_ADMIN_EMAIL` + `POCKETBASE_ADMIN_PASSWORD` — same values as your Coolify env
   - `RENDER_TRIGGER_KEY` — shared secret for `/api/workers/register`
   - `STORAGE_PROVIDERS_ENC_KEY` — 32-byte hex; must match your dashboard

   NIM / Groq / Pexels / HuggingFace / YouTube OAuth all live in Pocketbase's `api_keys` collection — manage them via the dashboard's **Connections** page. This notebook never touches them.
3. **Toggle notebook access ON** for each secret (the X icons → green dots).
4. Run cells top to bottom. **Cell 4 is the GPU diagnostic** — make sure it prints `READY: h264_nvenc` before proceeding; otherwise the render falls back to CPU and will be slow.
5. The last cell is a blocking server loop — leave it running.

In [ ]:
# 1) System deps
import subprocess, sys, os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.check_call(["apt-get", "-qq", "update"])
# fonts-noto{,-cjk,-color-emoji} + fonts-dejavu-core — needed for
# burned-in captions to render non-Latin scripts (Devanagari, Bengali,
# CJK, Arabic, Thai). DejaVu Sans is the primary font (bundled default);
# libass falls back to Noto CJK for CJK/emoji.
subprocess.check_call(["apt-get", "-qq", "install", "-y",
                       "--no-install-recommends",
                       "ffmpeg", "fonts-dejavu-core",
                       "fonts-noto", "fonts-noto-cjk",
                       "fonts-noto-color-emoji"])
subprocess.run(["fc-cache", "-f"], check=False)
print("ffmpeg:", subprocess.check_output(["ffmpeg", "-version"]).decode().splitlines()[0])

In [ ]:
# 2) Clone the repo (or pull latest if already present)
REPO_URL = "https://github.com/Ahsan3301/yt_agent.git"
BRANCH   = "main"
import os, subprocess
if not os.path.exists('/content/yt_agent'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, '/content/yt_agent'])
else:
    subprocess.check_call(['git', '-C', '/content/yt_agent', 'pull', '--ff-only'])
os.chdir('/content/yt_agent')
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3) Python deps — batched where safe, split when a per-package flag
#    would otherwise leak to every package in the same pip call.
import subprocess, sys, os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'espeak-ng'])
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'phonemizer'], check=False)

# MAIN BATCH — no per-package flags.
# transformers <4.50 (Kokoro AlbertModel) + diffusers <0.32 (flux2's
# Qwen3ForCausalLM requires transformers>=4.50, so we can't have both).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '-r', 'requirements.txt',
    '-r', 'requirements-browser.txt',
    'decord==0.6.0', 'av==12.3.0',
    'hf_transfer',
    'espeakng-loader>=0.2',
    'soundfile>=0.12',
    'kokoro>=0.9.4', 'misaki>=0.9.4',
    'transformers>=4.44,<4.50', 'accelerate>=0.30', 'safetensors>=0.4',
    'huggingface_hub>=0.23', 'peft>=0.10',
    'pyOpenSSL>=25.0.0', 'cryptography>=45.0.0',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--force-reinstall', 'phonemizer-fork>=3.3.2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--upgrade', 'edge-tts>=7.0.0'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'diffusers>=0.30,<0.32'])

_playwright_log = open('/tmp/playwright-install.log', 'w')
_playwright_bg = subprocess.Popen(
    [sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
    stdout=_playwright_log, stderr=subprocess.STDOUT,
)
print(f'playwright chromium install running in background (pid={_playwright_bg.pid})', flush=True)

import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
for _mod, _label in [('torchvision','torchvision'),('transformers','transformers'),('diffusers','diffusers')]:
    try:
        _m = __import__(_mod)
        print(f'{_label}:', _m.__version__)
    except Exception as e:
        print(f'{_label}: NOT ready ->', e)
try:
    from kokoro import KPipeline
    print('kokoro: ready')
except Exception as e:
    print('kokoro: NOT ready ->', e)
try:
    from diffusers import AutoPipelineForText2Image  # noqa: F401
    print('diffusers AutoPipeline: ready (local_sdxl provider available)')
except Exception as e:
    print('diffusers AutoPipeline: NOT ready ->', e)
try:
    import edge_tts
    print('edge-tts:', edge_tts.__version__ if hasattr(edge_tts, '__version__') else 'installed')
except Exception as e:
    print('edge-tts: NOT ready ->', e)
try:
    import OpenSSL, cryptography
    print(f'crypto: pyOpenSSL={OpenSSL.__version__} cryptography={cryptography.__version__}')
except Exception as e:
    print('crypto: NOT ready ->', e)

In [ ]:
# 4) GPU / NVENC diagnostic
#
# Three checks decide whether the pipeline will render on GPU (5-10x
# faster) or fall back to CPU:
#   A. nvidia-smi sees a GPU
#   B. the installed ffmpeg was built with h264_nvenc support
#   C. a live NVENC smoke encode actually succeeds at runtime
#
# Note on the smoke size: NVENC has a minimum frame size (~145x49 for
# H.264). The previous 64x64 test failed even on working T4s with
# "Frame Dimension less than the minimum supported value" — that's a
# test bug, not a GPU problem. 320x240 is safely above the minimum and
# tells the truth about whether your render pipeline will work.
#
# If you see 'READY: h264_nvenc' at the bottom, the backend's encoder
# picker (modules/editor._detect_nvenc) will pick GPU on startup and
# the Monitor card will show 'renders: GPU ✓'.
import subprocess, shutil, os

print('A) nvidia-smi -L')
if shutil.which('nvidia-smi') is None:
    print('   FAIL: nvidia-smi not on PATH — runtime is probably CPU. Runtime → Change runtime type → T4 GPU.')
    gpu_ok = False
else:
    r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=8)
    print('  ', (r.stdout or '').strip() or r.stderr.strip())
    gpu_ok = (r.returncode == 0 and 'GPU' in (r.stdout or ''))

print()
print('B) ffmpeg -encoders | grep h264_nvenc')
r = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True, timeout=8)
ffmpeg_ok = 'h264_nvenc' in (r.stdout or '')
print('   PASS' if ffmpeg_ok else '   FAIL: the installed ffmpeg has no h264_nvenc encoder — install a CUDA-enabled build')

print()
print('C) live NVENC smoke encode (320x240 black, 0.1s)')
smoke_ok = False
if gpu_ok and ffmpeg_ok:
    r = subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error',
        '-f', 'lavfi', '-i', 'color=c=black:s=320x240:d=0.1',
        '-c:v', 'h264_nvenc', '-f', 'null', '-',
    ], capture_output=True, text=True, timeout=15)
    smoke_ok = (r.returncode == 0)
    if smoke_ok:
        print('   PASS')
    else:
        print('   FAIL:', (r.stderr or '').strip()[-300:])
else:
    print('   SKIPPED (A or B failed)')

print()
if gpu_ok and ffmpeg_ok and smoke_ok:
    print('READY: h264_nvenc — pipeline will render on GPU')
    os.environ.pop('FFMPEG_FORCE_CPU', None)
else:
    print('READY: libx264 (CPU only) — to fix, change runtime to T4 GPU and run all cells again')
    # We don't set FFMPEG_FORCE_CPU here — the backend's auto-detect
    # falls back to libx264 on its own. Setting it would also stop
    # any later in-place fixes from being picked up after restart.


In [ ]:
# 5) Bootstrap secrets.
#
#    REQUIRED (Pocketbase + Coolify mode — the current default):
#      COOLIFY_BASE_URL             (dashboard URL, e.g. https://yt-agent.thyker.online)
#      PB_URL                       (Pocketbase URL, usually <dashboard>/pb)
#      POCKETBASE_ADMIN_EMAIL       (superuser email)
#      POCKETBASE_ADMIN_PASSWORD    (superuser password)
#      RENDER_TRIGGER_KEY           (shared secret for /api/workers/register etc.)
#      STORAGE_PROVIDERS_ENC_KEY    (32-byte hex, same value as your dashboard)
#
#    OPTIONAL (legacy Firestore path — only if your dashboard is still
#    on Vercel + Firebase):
#      GOOGLE_APPLICATION_CREDENTIALS_JSON      (raw JSON)
#      GOOGLE_APPLICATION_CREDENTIALS_JSON_B64  (base64 alt — mirrors Kaggle's Secrets workaround)
#      R2_* / SFTP_* / PUBLIC_BASE_URL          (used if no PB storage provider configured)
#
#    NIM / Groq / Pexels / HuggingFace etc. all live in Pocketbase's
#    api_keys collection — manage via the dashboard's Connections page.
from google.colab import userdata
import os

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'R2_ACCOUNT_ID', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY',
    'R2_BUCKET', 'R2_PUBLIC_URL', 'R2_MAX_GB',
    'SFTP_HOST', 'SFTP_PORT', 'SFTP_USER', 'SFTP_PASS', 'SFTP_BASE_DIR',
    'PUBLIC_BASE_URL',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]

missing = []
for k in REQUIRED + OPTIONAL:
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = str(v)
    except Exception:
        if k in REQUIRED:
            missing.append(k)
        continue
    if k in REQUIRED and not os.environ.get(k):
        missing.append(k)

# Defaults for the outbound-poll model.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'colab-gpu')
# Idle auto-shutdown DISABLED for Colab — the user wants the runtime
# to keep running until they manually stop it. Setting the shutdown
# var to -1 makes backend/idle_watchdog.py check IDLE_TIMEOUT_SECONDS
# <= 0 and skip starting the watchdog thread entirely (see
# backend/idle_watchdog.py:165). Kaggle keeps the 25 min default via
# its own env var; only Colab is exempted here.
#
# Caveats we can't override from code:
#   • Colab's browser-side idle timeout kicks after ~90 min of no
#     activity on the tab. Cell 8 injects a JS keep-alive that pings
#     the connect button every minute to keep the session live.
#   • Colab free tier still has a ~12 hr HARD session cap regardless
#     of activity — that's a Google limit we can't circumvent.
os.environ['KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS'] = '-1'
os.environ['IDLE_TIMEOUT_SECONDS'] = '-1'

if missing:
    print('MISSING REQUIRED BOOTSTRAP SECRETS:', ', '.join(missing))
    print('Add them in the Secrets panel and toggle notebook access ON, then re-run this cell.')
else:
    print('Bootstrap secrets loaded.')
    print('  Pocketbase     :', os.environ.get('PB_URL'))
    print('  Dashboard API  :', os.environ.get('COOLIFY_BASE_URL'))
    print('  Worker mode    :', os.environ.get('WORKER_MODE'))
    print('  DB backend     :', os.environ.get('DB_BACKEND'))
    print('  Storage backend:', os.environ.get('STORAGE_BACKEND'))
    print('  Tier / label   :', os.environ.get('INSTANCE_TIER'), '/', os.environ.get('INSTANCE_LABEL'))
    print('  Idle shutdown  : DISABLED (never auto-terminate)')
    print('  Storage providers + API keys will be pulled from Pocketbase.')

In [ ]:
# 6) (Skipped in outbound-poll mode.) In WORKER_MODE=outbound_poll the
#    worker calls the dashboard from inside Colab — no inbound tunnel
#    is needed. The cell below is kept for legacy WORKER_MODE=tunnel
#    deployments (Vercel + Firestore) and is a no-op otherwise.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL — check /tmp/cloudflared.log')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 6.5) Pre-warm SDXL weights in the BACKGROUND — doesn't block boot.
#
# Only downloads the fp16 variants. Our shotfinder loads with
# variant="fp16" so the fp32 .safetensors files (unet 10.3G, text
# encoders 3G+, top-level bundle 13.9G) are pure waste on Colab's
# slow HF bandwidth. Narrow the allow_patterns to fp16-only + config
# files → ~7 GB instead of ~35 GB.
import os, time, subprocess, threading
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '180')

def _bg_snapshot_sdxl():
    try:
        t0 = time.time()
        print('preboot(bg): snapshotting SDXL fp16 weights into HF cache...', flush=True)
        from huggingface_hub import snapshot_download
        _sdxl_model = os.environ.get('LOCAL_SDXL_MODEL', 'stabilityai/sdxl-turbo')
        snapshot_download(
            _sdxl_model,
            allow_patterns=[
                # Configs + tokenizer json/text files.
                '*.json', '*.txt',
                # fp16 model weights only (skips ~28 GB of fp32 dupes).
                '**/*.fp16.safetensors',
                'sd_xl_turbo_1.0_fp16.safetensors',
            ],
        )
        print(f'preboot(bg): SDXL fp16 weights cached in {time.time()-t0:.1f}s', flush=True)
    except Exception as e:
        print(f'preboot(bg): SDXL cache warm skipped ({e!r})', flush=True)

threading.Thread(target=_bg_snapshot_sdxl, name='sdxl-snapshot', daemon=True).start()
print('preboot: SDXL snapshot kicked in background thread', flush=True)

try:
    out = subprocess.check_output(['nvidia-smi', '-L'], timeout=5).decode().strip()
    print('preboot: nvidia-smi -L:\n' + out, flush=True)
except Exception as e:
    print(f'preboot: nvidia-smi -L failed ({e!r})', flush=True)

print('preboot: done (main thread not blocked on SDXL)', flush=True)

In [ ]:
# 6.6) Self-check — verify multi-GPU + multilingual wiring
# Wrapped in try/except so any check crash still lets uvicorn boot.
try:
    from backend import self_check
    print(self_check.run(text=True), flush=True)
except Exception as e:
    print(f'self_check skipped ({e!r})', flush=True)

In [ ]:
# 6.7) Colab browser keep-alive.
#
# Colab disconnects the runtime after ~90 min of BROWSER inactivity
# (nothing to do with our worker being busy — Google measures whether
# YOU are looking at the tab). This JS snippet clicks the connect
# button once a minute so Google thinks you're still there.
#
# Doesn't defeat the ~12 hr hard session cap on free tier — that's
# Google's non-negotiable limit for the free runtime. If you want
# past 12 hr, upgrade to Colab Pro or use Kaggle (30 GPU-hr/week
# on T4x2 rolling).
from IPython.display import Javascript, display
display(Javascript('''
(function() {
  if (window.__yt_agent_keepalive) return;
  window.__yt_agent_keepalive = setInterval(function() {
    try {
      var btn = document.querySelector('colab-connect-button');
      if (btn && btn.shadowRoot) {
        var inner = btn.shadowRoot.querySelector('#connect');
        if (inner) inner.click();
      }
      // Backup: click any visible "Reconnect" prompt if it appears.
      var reconnect = document.querySelector('colab-dialog paper-button[dialog-confirm]');
      if (reconnect) reconnect.click();
    } catch (e) { /* ignore, best-effort */ }
  }, 60 * 1000);
  console.log('yt-agent keep-alive installed (60s interval)');
})();
'''))
print('Colab browser keep-alive installed (clicks connect button every 60s).')

In [ ]:
# 7) Boot the backend. This cell BLOCKS — keep it running for the session.
#    In outbound-poll mode the worker POSTs to <COOLIFY_BASE_URL>/api/workers/register
#    on boot and heartbeats every 30s so the Monitor detects it. It then
#    polls /api/jobs/claim for queued renders. Watch the first
#    'video encoder: ...' log line — it tells you whether NVENC was picked.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])